# Log-Linear Regression Imputation for Requirements

This notebook implements a log-linear regression model to impute missing `requirements` values in the UNOCHA humanitarian data.

**Predictor Variables:**
* Country (categorical)
* Crisis type (categorical)
* Year (numeric)
* People in need (numeric, log-transformed)
* Country size / population (numeric, log-transformed)

**Approach:**
1. Load data from `workspace.unochadatasets.fts_requirements_funding_global`
2. Join with people in need and population data from `workspace.unochadatasets.hpc_hno_2025`
3. Split data into complete cases (training) and missing cases (to impute)
4. Encode categorical variables and log-transform numeric features
5. Train linear regression on log(requirements)
6. Predict missing values and transform back from log space
7. Create final dataset with imputed values

In [0]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt

In [0]:
# Load the requirements and funding data
df = spark.table("workspace.unochadatasets.fts_requirements_funding_global").toPandas()

print(f"Total rows: {len(df)}")
print(f"Missing requirements: {df['requirements'].isna().sum()}")
print(f"Non-missing requirements: {df['requirements'].notna().sum()}")
print(f"\nData shape: {df.shape}")
print(f"\nColumn types:\n{df.dtypes}")
df.head()

In [0]:
# Filter to only include years 2015-2025
print(f"Before filtering - Total rows: {len(df)}")
print(f"Year range: {df['year'].min()} to {df['year'].max()}")

df = df[(df['year'] >= 2015) & (df['year'] <= 2025)].copy()

print(f"\nAfter filtering to 2015-2025:")
print(f"  Total rows: {len(df)}")
print(f"  Missing requirements: {df['requirements'].isna().sum()}")
print(f"  Non-missing requirements: {df['requirements'].notna().sum()}")
print(f"  Year range: {df['year'].min()} to {df['year'].max()}")

In [0]:
# Load people in need data from hpc_hno_2025
df_pin = spark.table("workspace.unochadatasets.hpc_hno_2025").toPandas()

print(f"People in need data shape: {df_pin.shape}")
print(f"\nOriginal columns: {df_pin.columns.tolist()}")

# Clean column names (remove special characters)
df_pin.columns = df_pin.columns.str.replace('#', '').str.replace('+', '_').str.strip()

print(f"\nCleaned columns: {df_pin.columns.tolist()}")

# Convert numeric columns
df_pin['In Need'] = pd.to_numeric(df_pin['In Need'], errors='coerce')
df_pin['Population'] = pd.to_numeric(df_pin['Population'], errors='coerce')

# Aggregate by country: sum people in need, average population
df_pin_agg = df_pin.groupby('Country ISO3').agg({
    'In Need': 'sum',
    'Population': 'mean'  # Use mean to get country-level population estimate
}).reset_index()

df_pin_agg.columns = ['countryCode', 'people_in_need', 'country_size']

print(f"\nAggregated data shape: {df_pin_agg.shape}")
print(f"\nSample:")
print(df_pin_agg.head(10))

# Join with requirements data
df = df.merge(df_pin_agg, on='countryCode', how='left')

print(f"\nAfter joining people in need and country size:")
print(f"  Total rows: {len(df)}")
print(f"  Rows with people_in_need: {df['people_in_need'].notna().sum()}")
print(f"  Rows with country_size: {df['country_size'].notna().sum()}")
print(f"  Rows missing both: {(df['people_in_need'].isna() & df['country_size'].isna()).sum()}")

# Fill missing values with medians
df['people_in_need'] = df['people_in_need'].fillna(df['people_in_need'].median())
df['country_size'] = df['country_size'].fillna(df['country_size'].median())

print(f"\nAfter filling missing values with median:")
print(f"  people_in_need missing: {df['people_in_need'].isna().sum()}")
print(f"  country_size missing: {df['country_size'].isna().sum()}")

In [0]:
# Handle duplicate columns if they exist
if 'people_in_need_y' in df.columns:
    df['people_in_need'] = df['people_in_need_y']
    df = df.drop(['people_in_need_x', 'people_in_need_y'], axis=1, errors='ignore')
if 'country_size_y' in df.columns:
    df['country_size'] = df['country_size_y']
    df = df.drop(['country_size_x', 'country_size_y'], axis=1, errors='ignore')

# Separate complete cases (for training) and missing cases (to impute)
df_complete = df[df['requirements'].notna()].copy()
df_missing = df[df['requirements'].isna()].copy()

print(f"Complete cases (training set): {len(df_complete)}")
print(f"Missing cases (to impute): {len(df_missing)}")

# Check for invalid values (requirements <= 0) which can't be log-transformed
df_complete = df_complete[df_complete['requirements'] > 0]
print(f"Complete cases after removing non-positive values: {len(df_complete)}")

# Handle missing values in predictor variables
print(f"\nMissing values in predictors (complete cases):")
print(f"  countryCode: {df_complete['countryCode'].isna().sum()}")
print(f"  typeName: {df_complete['typeName'].isna().sum()}")
print(f"  year: {df_complete['year'].isna().sum()}")
print(f"  people_in_need: {df_complete['people_in_need'].isna().sum()}")
print(f"  country_size: {df_complete['country_size'].isna().sum()}")

# Fill missing categorical values with 'Unknown'
df_complete['countryCode'] = df_complete['countryCode'].fillna('Unknown')
df_complete['typeName'] = df_complete['typeName'].fillna('Unknown')
df_complete['year'] = df_complete['year'].fillna(0)

# Fill missing numeric values with medians
pin_median = df_complete['people_in_need'].median()
size_median = df_complete['country_size'].median()
df_complete['people_in_need'] = df_complete['people_in_need'].fillna(pin_median)
df_complete['country_size'] = df_complete['country_size'].fillna(size_median)

print(f"\nAfter filling missing values:")
print(f"  people_in_need missing: {df_complete['people_in_need'].isna().sum()}")
print(f"  country_size missing: {df_complete['country_size'].isna().sum()}")

In [0]:
# Encode categorical variables
le_country = LabelEncoder()
le_crisis = LabelEncoder()

# Fit encoders on complete cases
df_complete['country_encoded'] = le_country.fit_transform(df_complete['countryCode'])
df_complete['crisis_encoded'] = le_crisis.fit_transform(df_complete['typeName'])

# Create log-transformed target variable
df_complete['log_requirements'] = np.log(df_complete['requirements'])

# Log-transform numeric features to handle scale differences
df_complete['log_people_in_need'] = np.log(df_complete['people_in_need'] + 1)  # +1 to avoid log(0)
df_complete['log_country_size'] = np.log(df_complete['country_size'] + 1)

# Select features for the model (now includes country size)
feature_cols = ['country_encoded', 'crisis_encoded', 'year', 'log_people_in_need', 'log_country_size']
X_train = df_complete[feature_cols]
y_train = df_complete['log_requirements']

print(f"Training features shape: {X_train.shape}")
print(f"Target variable shape: {y_train.shape}")
print(f"\nFeature summary:")
print(X_train.describe())

## Model Improvement Strategies

The sections below show different approaches to improve the R² (currently 0.2350):

1. **Interaction terms**: Capture multiplicative relationships between features
2. **Advanced models**: Try Random Forest or Gradient Boosting for non-linear patterns
3. **Additional features**: Historical trends, regional effects
4. **Cross-validation**: Ensure model generalizes well

In [0]:
# Train baseline model first
model = LinearRegression()
model.fit(X_train, y_train)
train_score = model.score(X_train, y_train)

# Create interaction terms
df_complete['pin_x_size'] = df_complete['log_people_in_need'] * df_complete['log_country_size']
df_complete['pin_x_year'] = df_complete['log_people_in_need'] * df_complete['year']
df_complete['crisis_x_year'] = df_complete['crisis_encoded'] * df_complete['year']

# Add to feature set
feature_cols_enhanced = feature_cols + ['pin_x_size', 'pin_x_year', 'crisis_x_year']
X_train_enhanced = df_complete[feature_cols_enhanced]

# Train enhanced model
model_enhanced = LinearRegression()
model_enhanced.fit(X_train_enhanced, y_train)

train_score_enhanced = model_enhanced.score(X_train_enhanced, y_train)
print(f"Original R²: {train_score:.4f}")
print(f"Enhanced R²: {train_score_enhanced:.4f}")
print(f"Improvement: {(train_score_enhanced - train_score) * 100:.2f}%")

print(f"\nNew feature coefficients:")
for i, col in enumerate(feature_cols_enhanced[-3:]):
    print(f"  {col}: {model_enhanced.coef_[-(3-i)]:.4f}")

In [0]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

# Train Random Forest
rf_model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

rf_score = rf_model.score(X_train, y_train)
print(f"Linear Regression R²: {train_score:.4f}")
print(f"Random Forest R²: {rf_score:.4f}")

# Feature importance
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print(f"\nFeature Importance (Random Forest):")
print(feature_importance)

# Cross-validation to check generalization
cv_scores = cross_val_score(rf_model, X_train, y_train, cv=5, scoring='r2')
print(f"\nCross-validation R² scores: {cv_scores}")
print(f"Mean CV R²: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")

In [0]:
# Add lag features: previous year's requirements for same country
df_with_history = df.sort_values(['countryCode', 'year'])
df_with_history['prev_year_req'] = df_with_history.groupby('countryCode')['requirements'].shift(1)
df_with_history['prev_2year_req'] = df_with_history.groupby('countryCode')['requirements'].shift(2)

# Add rolling average
df_with_history['rolling_avg_req'] = df_with_history.groupby('countryCode')['requirements'].transform(
    lambda x: x.rolling(window=3, min_periods=1).mean().shift(1)
)

print("Historical features added:")
print(f"  prev_year_req: {df_with_history['prev_year_req'].notna().sum()} non-null values")
print(f"  prev_2year_req: {df_with_history['prev_2year_req'].notna().sum()} non-null values")
print(f"  rolling_avg_req: {df_with_history['rolling_avg_req'].notna().sum()} non-null values")

print("\nNote: You can merge these back and retrain with these additional features")

In [0]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split

# Split data for proper validation
X_train_split, X_val, y_train_split, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

# Compare multiple models
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=42)
}

results = []
for name, model_test in models.items():
    model_test.fit(X_train_split, y_train_split)
    train_r2 = model_test.score(X_train_split, y_train_split)
    val_r2 = model_test.score(X_val, y_val)
    results.append({
        'Model': name,
        'Train R²': train_r2,
        'Validation R²': val_r2,
        'Overfit Gap': train_r2 - val_r2
    })

results_df = pd.DataFrame(results)
print("Model Comparison:")
print(results_df.to_string(index=False))
print("\nLower 'Overfit Gap' indicates better generalization")

## Other Strategies to Consider

**Data Quality:**
* Remove outliers using z-score or IQR methods
* Investigate patterns in residuals to identify systematic errors
* Stratify models by crisis type (different model for each type)

**Feature Engineering:**
* Regional groupings (Middle East, Africa, Asia, etc.)
* GDP per capita or economic indicators
* Conflict intensity metrics
* Duration of crisis (endDate - startDate)
* Number of affected areas/provinces

**Advanced Techniques:**
* XGBoost for better gradient boosting
* Neural networks for complex patterns
* Stacking/Blending multiple models
* Regularization (Ridge/Lasso) to prevent overfitting

**Domain Knowledge:**
* Talk to humanitarian experts about what drives requirements
* Identify if certain crisis types have different drivers
* Consider temporal effects (COVID-19 era vs. pre-COVID)

In [0]:
# Train the linear regression model on log-transformed target
model = LinearRegression()
model.fit(X_train, y_train)

print("Model trained successfully!")
print(f"\nModel coefficients:")
for i, col in enumerate(feature_cols):
    print(f"  {col}: {model.coef_[i]:.4f}")
print(f"\nIntercept: {model.intercept_:.4f}")

# Calculate R-squared on training data
train_score = model.score(X_train, y_train)
print(f"\nTraining R²: {train_score:.4f}")

In [0]:
from sklearn.ensemble import GradientBoostingRegressor

# Train Gradient Boosting on full training data
gb_model = GradientBoostingRegressor(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42,
    verbose=0
)

gb_model.fit(X_train, y_train)

gb_score = gb_model.score(X_train, y_train)
print("Final Model: Gradient Boosting Regressor")
print(f"\nTraining R²: {gb_score:.4f}")
print(f"Linear Regression R² (for comparison): {train_score:.4f}")
print(f"\nImprovement: {((gb_score - train_score) / train_score * 100):.1f}%")

# Feature importance
feature_importance_gb = pd.DataFrame({
    'feature': feature_cols,
    'importance': gb_model.feature_importances_
}).sort_values('importance', ascending=False)

print(f"\nFeature Importance:")
for idx, row in feature_importance_gb.iterrows():
    print(f"  {row['feature']}: {row['importance']:.4f}")

In [0]:
# Prepare missing cases for prediction
df_missing['countryCode'] = df_missing['countryCode'].fillna('Unknown')
df_missing['typeName'] = df_missing['typeName'].fillna('Unknown')
df_missing['year'] = df_missing['year'].fillna(0)

# Fill missing numeric values with medians
pin_median = df_missing['people_in_need'].median()
size_median = df_missing['country_size'].median()
df_missing['people_in_need'] = df_missing['people_in_need'].fillna(pin_median)
df_missing['country_size'] = df_missing['country_size'].fillna(size_median)

# Handle unseen categories by using 'Unknown' or the most common category
def encode_with_unknown(value, encoder, default=0):
    try:
        return encoder.transform([value])[0]
    except ValueError:
        # If category not seen in training, use default (or most common)
        return default

df_missing['country_encoded'] = df_missing['countryCode'].apply(
    lambda x: encode_with_unknown(x, le_country, default=le_country.transform(['Unknown'])[0] if 'Unknown' in le_country.classes_ else 0)
)
df_missing['crisis_encoded'] = df_missing['typeName'].apply(
    lambda x: encode_with_unknown(x, le_crisis, default=le_crisis.transform(['Unknown'])[0] if 'Unknown' in le_crisis.classes_ else 0)
)

# Transform numeric features to log scale
df_missing['log_people_in_need'] = np.log(df_missing['people_in_need'] + 1)
df_missing['log_country_size'] = np.log(df_missing['country_size'] + 1)

print(f"Missing cases prepared for prediction: {len(df_missing)}")
print(f"\nMissing data summary:")
print(df_missing[['countryCode', 'typeName', 'year', 'people_in_need', 'country_size']].describe())
print(f"\nMissing values remaining:")
print(f"  people_in_need: {df_missing['people_in_need'].isna().sum()}")
print(f"  country_size: {df_missing['country_size'].isna().sum()}")

In [0]:
# Make predictions on missing cases using Gradient Boosting
X_missing = df_missing[feature_cols]
log_predictions = gb_model.predict(X_missing)  # Using Gradient Boosting model

# Transform back from log space
predictions = np.exp(log_predictions)

# Add predictions to the missing data
df_missing['requirements_imputed'] = predictions
df_missing['imputation_flag'] = True

print(f"Predictions generated for {len(predictions)} missing values")
print(f"Model used: Gradient Boosting Regressor")
print(f"\nImputed requirements summary:")
print(df_missing['requirements_imputed'].describe())
print(f"\nSample imputed values (showing all predictors):")
print(df_missing[['countryCode', 'typeName', 'year', 'people_in_need', 'country_size', 'requirements_imputed']].head(10))

In [0]:
# Mark complete cases
df_complete['requirements_imputed'] = df_complete['requirements']
df_complete['imputation_flag'] = False

# Combine complete and imputed data
df_final = pd.concat([
    df_complete[['countryCode', 'id', 'name', 'code', 'typeId', 'typeName', 
                 'startDate', 'endDate', 'year', 'requirements', 'requirements_imputed', 
                 'imputation_flag', 'funding', 'percentFunded', 'people_in_need', 'country_size']],
    df_missing[['countryCode', 'id', 'name', 'code', 'typeId', 'typeName', 
                'startDate', 'endDate', 'year', 'requirements', 'requirements_imputed', 
                'imputation_flag', 'funding', 'percentFunded', 'people_in_need', 'country_size']]
], ignore_index=True)

print(f"Final dataset shape: {df_final.shape}")
print(f"\nImputation summary:")
print(f"  Original non-missing: {(~df_final['imputation_flag']).sum()}")
print(f"  Imputed values: {df_final['imputation_flag'].sum()}")
print(f"  Total: {len(df_final)}")

display(df_final.head(20))

In [0]:
# Create visualization of imputed vs original values
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Distribution comparison
axes[0].hist(df_complete['requirements'], bins=50, alpha=0.5, label='Original', edgecolor='black')
axes[0].hist(df_missing['requirements_imputed'], bins=50, alpha=0.5, label='Imputed', edgecolor='black')
axes[0].set_xlabel('Requirements')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Original vs Imputed Requirements')
axes[0].legend()
axes[0].set_yscale('log')

# Plot 2: Log-scale distribution
axes[1].hist(np.log(df_complete['requirements']), bins=50, alpha=0.5, label='Original (log)', edgecolor='black')
axes[1].hist(np.log(df_missing['requirements_imputed']), bins=50, alpha=0.5, label='Imputed (log)', edgecolor='black')
axes[1].set_xlabel('Log(Requirements)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Log-Transformed Requirements')
axes[1].legend()

plt.tight_layout()
plt.show()

print("\nVisualization complete!")

## Critical Analysis: Can We Trust These Imputations?

### 🔴 Major Concerns

**1. Severe Data Imbalance**
* **68% of data is missing** (2,607 out of 3,805 rows)
* Only 1,194 complete cases available for training (31%)
* We're imputing MORE data than we have to train on

**2. Significant Model Overfitting**
* **Random Forest**: Training R² = 0.87 but Cross-validation R² = **0.17** ❌
* **Gradient Boosting** (final model): Training R² = 0.89, Validation R² = 0.71 (gap of 0.18)
* **Linear Regression**: Training R² = 0.23, Validation R² = 0.24 (best generalization!) ✓
* The complex models memorize training data but don't generalize well

**3. Category Mismatch Problem**
* Many missing records have `typeName='Unknown'`
* The model was NEVER trained on 'Unknown' crisis types
* These records get assigned arbitrary encoded values, producing unreliable predictions
* **First 9 Afghanistan records have IDENTICAL predictions** because they all have the same (invalid) inputs

**4. Distribution Mismatch**
* Imputed values: Mean = $267M, Median = $153M, Max = $3.3B
* Need to compare with original distribution to verify reasonableness
* Some imputed values (e.g., $1.19B for Afghanistan 'Unknown' type) seem suspiciously high

**5. Final Model Has No Validation**
* The Gradient Boosting model was trained on ALL training data
* No held-out validation set for the final model
* We don't know its true generalization performance on unseen data

### 🟡 Moderate Concerns

**6. Feature Importance Imbalance**
* `country_encoded`: 35% importance (may not generalize to countries with sparse data)
* `crisis_encoded`: 22% importance (but many missing records have invalid crisis types)
* `log_country_size`: Only 1.9% importance (barely contributes)

**7. Limited Predictive Power**
* Best realistic R² is around 0.24 (Linear Regression) or 0.71 (optimistic Gradient Boosting validation)
* This means **76-29% of variance remains unexplained**
* Many other factors drive requirements that aren't captured

### ✅ What Works

* Log transformation of target variable is appropriate
* Feature selection is reasonable (country, crisis type, year, people in need, population)
* Linear Regression shows good generalization despite low R²
* Gradient Boosting captures non-linear patterns when data is available

### 📊 Recommendation

**Option 1: Conservative Approach (RECOMMENDED)**
* Use **Linear Regression** despite lower R² (0.24) because it generalizes well
* Flag all imputations with confidence intervals
* Investigate missing records with 'Unknown' crisis types separately
* Consider imputing ONLY for records with known crisis types

**Option 2: Tiered Approach**
* Use Gradient Boosting for records with complete, known crisis types
* Use Linear Regression for records with 'Unknown' or sparse data
* Add a confidence score based on data quality

**Option 3: Don't Impute**
* With 68% missing and severe overfitting, consider whether imputation adds value
* Use only the 1,194 complete cases for analysis
* Document the missingness pattern as a finding rather than filling it

### 🎯 Next Steps If Proceeding

1. **Fix category encoding**: Handle 'Unknown' crisis types properly
2. **Add confidence intervals**: Report prediction uncertainty
3. **Validate distribution**: Compare imputed vs original distributions statistically
4. **Investigate missingness**: Why are 68% of requirements missing? Is it systematic?
5. **Use simpler model**: Linear Regression may be more appropriate
6. **Get domain expertise**: Consult humanitarian experts on what drives requirements

In [0]:
# Quantify the critical concerns identified above
import scipy.stats as stats

print("=" * 70)
print("1. DATA IMBALANCE")
print("=" * 70)
print(f"Training data: {len(df_complete)} rows ({len(df_complete)/len(df)*100:.1f}%)")
print(f"Missing data to impute: {len(df_missing)} rows ({len(df_missing)/len(df)*100:.1f}%)")
print(f"\n⚠️  We're imputing {len(df_missing)/len(df_complete):.2f}x MORE data than we trained on!\n")

print("=" * 70)
print("2. DISTRIBUTION COMPARISON")
print("=" * 70)
print(f"\nOriginal Requirements:")
print(f"  Mean:   ${df_complete['requirements'].mean():,.0f}")
print(f"  Median: ${df_complete['requirements'].median():,.0f}")
print(f"  Std:    ${df_complete['requirements'].std():,.0f}")
print(f"  Min:    ${df_complete['requirements'].min():,.0f}")
print(f"  Max:    ${df_complete['requirements'].max():,.0f}")

print(f"\nImputed Requirements:")
print(f"  Mean:   ${df_missing['requirements_imputed'].mean():,.0f}")
print(f"  Median: ${df_missing['requirements_imputed'].median():,.0f}")
print(f"  Std:    ${df_missing['requirements_imputed'].std():,.0f}")
print(f"  Min:    ${df_missing['requirements_imputed'].min():,.0f}")
print(f"  Max:    ${df_missing['requirements_imputed'].max():,.0f}")

print(f"\nDifference:")
print(f"  Mean ratio: {df_missing['requirements_imputed'].mean() / df_complete['requirements'].mean():.2f}x")
print(f"  Median ratio: {df_missing['requirements_imputed'].median() / df_complete['requirements'].median():.2f}x")

# Statistical test for distribution similarity
ks_stat, ks_pval = stats.ks_2samp(
    np.log(df_complete['requirements']), 
    np.log(df_missing['requirements_imputed'])
)
print(f"\nKolmogorov-Smirnov test (log scale):")
print(f"  Statistic: {ks_stat:.4f}")
print(f"  P-value: {ks_pval:.4f}")
if ks_pval < 0.05:
    print(f"  ⚠️  Distributions are SIGNIFICANTLY DIFFERENT (p < 0.05)")
else:
    print(f"  ✓ Distributions are similar (p >= 0.05)")

print("\n" + "=" * 70)
print("3. 'UNKNOWN' CRISIS TYPE PROBLEM")
print("=" * 70)
unknown_count = (df_missing['typeName'] == 'Unknown').sum()
print(f"Missing records with 'Unknown' crisis type: {unknown_count} / {len(df_missing)}")
print(f"Percentage: {unknown_count/len(df_missing)*100:.1f}%")

if unknown_count > 0:
    print(f"\n⚠️  {unknown_count} records have invalid crisis types!")
    print(f"     These were never seen during training and will have unreliable predictions.")
    
    # Show duplicate predictions
    unknown_preds = df_missing[df_missing['typeName'] == 'Unknown']['requirements_imputed']
    unique_preds = unknown_preds.nunique()
    print(f"\nUnique predictions for 'Unknown' type: {unique_preds} / {unknown_count}")
    if unique_preds < unknown_count:
        print(f"⚠️  Many 'Unknown' records have IDENTICAL predictions (grouped by other features)")

print("\n" + "=" * 70)
print("4. CRISIS TYPE DISTRIBUTION")
print("=" * 70)
print("\nIn Training Data:")
print(df_complete['typeName'].value_counts())
print("\nIn Missing Data:")
print(df_missing['typeName'].value_counts())

print("\n" + "=" * 70)
print("5. MODEL RELIABILITY SUMMARY")
print("=" * 70)
print(f"\nBest Validation R²: 0.71 (Gradient Boosting)")
print(f"Best Generalization R²: 0.24 (Linear Regression)")
print(f"Overfitting Gap: 0.18 (Gradient Boosting)")
print(f"\nVariance Explained: 24-71% (depending on model)")
print(f"Variance Unexplained: 29-76%")
print(f"\n⚠️  Imputations have high uncertainty due to:")
print(f"     - Limited training data (1,194 cases)")
print(f"     - Model overfitting (training much better than validation)")
print(f"     - Invalid categories in missing data")
print(f"     - Low explained variance (most variation not captured)")